# Lab 01: RAG Pipeline from Scratch

**Course 06 — RAG: Retrieval Augmented Generation**

This lab demonstrates building a zero-framework modular RAG pipeline in pure Python using `OpenAI` embeddings and `LLMGateway` for grounded generation.

In [ ]:
import math
import os
import sys
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional

# Ensure project root is in python path
sys.path.append('..')
from labs.common.gateway import LLMGateway, OpenAIProvider
from openai import OpenAI

In [ ]:
DOCUMENTS = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a generator to produce grounded answers.",
    "Vector databases store dense embeddings and support fast nearest-neighbor similarity search. Popular options include Chroma, Pinecone, and pgvector.",
    "Chunking is the process of splitting documents into smaller pieces for retrieval. Common strategies include fixed-size, sentence-based, and recursive splitting.",
    "Cosine similarity measures the angle between two vectors. A score of 1.0 means identical direction; 0.0 means orthogonal.",
    "The embedding model converts text into dense numerical vectors. OpenAI's text-embedding-3-small produces 1536-dimensional vectors.",
    "Hybrid search combines keyword search (BM25) with dense vector search to improve recall. Reciprocal Rank Fusion (RRF) merges result lists.",
]

In [ ]:
@dataclass
class Chunk:
    id: int
    text: str
    vector: List[float]

class RAGPipeline:
    def __init__(self, gateway: Optional[LLMGateway] = None):
        self.gateway = gateway or LLMGateway([OpenAIProvider()])
        self.openai_client = OpenAI()
        self.chunks: List[Chunk] = []

    def get_embedding(self, text: str) -> List[float]:
        res = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return res.data[0].embedding

    def index(self, docs: List[str]) -> None:
        self.chunks = []
        for i, text in enumerate(docs):
            vec = self.get_embedding(text)
            self.chunks.append(Chunk(id=i + 1, text=text, vector=vec))

    @staticmethod
    def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
        dot = sum(a * b for a, b in zip(vec_a, vec_b))
        norm_a = math.sqrt(sum(a * a for a in vec_a))
        norm_b = math.sqrt(sum(b * b for b in vec_b))
        return dot / (norm_a * norm_b) if norm_a > 0 and norm_b > 0 else 0.0

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        q_vec = self.get_embedding(query)
        scored = []
        for c in self.chunks:
            score = self.cosine_similarity(q_vec, c.vector)
            scored.append((c, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def query(self, user_query: str) -> str:
        results = self.retrieve(user_query, top_k=3)
        context = "\n".join(f"[{c.id}] {c.text}" for c, _ in results)

        messages = [
            {"role": "system", "content": "Answer the question based ONLY on the context provided. Cite sources inline using [1], [2]."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_query}"}
        ]

        response = self.gateway.generate(messages=messages, temperature=0.0)
        return response.content

In [ ]:
print("--- Lab 01: Running RAG Pipeline ---")
pipeline = RAGPipeline()
pipeline.index(DOCUMENTS)

q = "What is RAG and how does it work?"
print(f"Query: {q}")
ans = pipeline.query(q)
print(f"\nAnswer:\n{ans}")